# End-to-End TF-IDF Based Retrieval and Evaluation Framework

# Step 1 — Environment Setup

This notebook implements a complete **TF-IDF based Retrieval System** with evaluation and comparison.

We import:

- `numpy` → vector algebra  
- `pandas` → evaluation reporting  
- `matplotlib` & `seaborn` → visualization  
- `sklearn` → baseline comparison  
- `logging` → structured debugging  

Logging allows controlled verbosity without flooding output.

In [93]:
import numpy as np
import re
import pandas as pd
import logging
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

# ---------- Logging Setup ----------
logging.basicConfig(
    level=logging.INFO,   
    format="%(levelname)s: %(message)s"
)

logger = logging.getLogger()
logger.info("Libraries Loaded.")

INFO: Libraries Loaded.


# Step 2 — Corpus Construction

We define the document collection as:

$$
D = \{ s_1, s_2, \dots, s_N \}
$$

Where:

- $D$ = document corpus  
- $s_i$ = i-th sentence  
- $N$ = total number of sentences  

The paragraph is split into individual sentences to form a retrieval dataset.

In [94]:
print("STEP 2: Loading Context...\n")

context = """What should one do? That may seem a strange question, but it's not meaningless or unanswerable. It's the sort of question kids ask before they learn not to ask big questions. I only came across it myself in the process of investigating something else. But once I did, I thought I should at least try to answer it.

So what should one do? One should help people, and take care of the world. Those two are obvious. But is there anything else? When I ask that, the answer that pops up is Make good new things.

I can't prove that one should do this, any more than I can prove that one should help people or take care of the world. We're talking about first principles here. But I can explain why this principle makes sense. The most impressive thing humans can do is to think. It may be the most impressive thing that can be done. And the best kind of thinking, or more precisely the best proof that one has thought well, is to make good new things.

I mean new things in a very general sense. Newton's physics was a good new thing. Indeed, the first version of this principle was to have good new ideas. But that didn't seem general enough: it didn't include making art or music, for example, except insofar as they embody new ideas. And while they may embody new ideas, that's not all they embody, unless you stretch the word "idea" so uselessly thin that it includes everything that goes through your nervous system.

Even for ideas that one has consciously, though, I prefer the phrasing "make good new things." There are other ways to describe the best kind of thinking. To make discoveries, for example, or to understand something more deeply than others have. But how well do you understand something if you can't make a model of it, or write about it? Indeed, trying to express what you understand is not just a way to prove that you understand it, but a way to understand it better.

Another reason I like this phrasing is that it biases us toward creation. It causes us to prefer the kind of ideas that are naturally seen as making things rather than, say, making critical observations about things other people have made. Those are ideas too, and sometimes valuable ones, but it's easy to trick oneself into believing they're more valuable than they are. Criticism seems sophisticated, and making new things often seems awkward, especially at first; and yet it's precisely those first steps that are most rare and valuable.

Is newness essential? I think so. Obviously it's essential in science. If you copied a paper of someone else's and published it as your own, it would seem not merely unimpressive but dishonest. And it's similar in the arts. A copy of a good painting can be a pleasing thing, but it's not impressive in the way the original was. Which in turn implies it's not impressive to make the same thing over and over, however well; you're just copying yourself.

Note though that we're talking about a different kind of should with this principle. Taking care of people and the world are shoulds in the sense that they're one's duty, but making good new things is a should in the sense that this is how to live to one's full potential. Historically most rules about how to live have been a mix of both kinds of should, though usually with more of the former than the latter. [1]

For most of history the question "What should one do?" got much the same answer everywhere, whether you asked Cicero or Confucius. You should be wise, brave, honest, temperate, and just, uphold tradition, and serve the public interest. There was a long stretch where in some parts of the world the answer became "Serve God," but in practice it was still considered good to be wise, brave, honest, temperate, and just, uphold tradition, and serve the public interest. And indeed this recipe would have seemed right to most Victorians. But there's nothing in it about taking care of the world or making new things, and that's a bit worrying, because it seems like this question should be a timeless one. The answer shouldn't change much.

I'm not too worried that the traditional answers don't mention taking care of the world. Obviously people only started to care about that once it became clear we could ruin it. But how can making good new things be important if the traditional answers don't mention it?

The traditional answers were answers to a slightly different question. They were answers to the question of how to be, rather than what to do. The audience didn't have a lot of choice about what to do. The audience up till recent centuries was the landowning class, which was also the political class. They weren't choosing between doing physics and writing novels. Their work was foreordained: manage their estates, participate in politics, fight when necessary. It was ok to do certain other kinds of work in one's spare time, but ideally one didn't have any. Cicero's De Officiis is one of the great classical answers to the question of how to live, and in it he explicitly says that he wouldn't even be writing it if he hadn't been excluded from public life by recent political upheavals. [2]

There were of course people doing what we would now call "original work," and they were often admired for it, but they weren't seen as models. Archimedes knew that he was the first to prove that a sphere has 2/3 the volume of the smallest enclosing cylinder and was very pleased about it. But you don't find ancient writers urging their readers to emulate him. They regarded him more as a prodigy than a model.

Now many more of us can follow Archimedes's example and devote most of our attention to one kind of work. He turned out to be a model after all, along with a collection of other people that his contemporaries would have found it strange to treat as a distinct group, because the vein of people making new things ran at right angles to the social hierarchy.

What kinds of new things count? I'd rather leave that question to the makers of them. It would be a risky business to try to define any kind of threshold, because new kinds of work are often despised at first. Raymond Chandler was writing literal pulp fiction, and he's now recognized as one of the best writers of the twentieth century. Indeed this pattern is so common that you can use it as a recipe: if you're excited about some kind of work that's not considered prestigious and you can explain what everyone else is overlooking about it, then this is not merely a kind of work that's ok to do, but one to seek out.

The other reason I wouldn't want to define any thresholds is that we don't need them. The kind of people who make good new things don't need rules to keep them honest.

So there's my guess at a set of principles to live by: take care of people and the world, and make good new things. Different people will do these to varying degrees. There will presumably be lots who focus entirely on taking care of people. There will be a few who focus mostly on making new things. But even if you're one of those, you should at least make sure that the new things you make don't net harm people or the world. And if you go a step further and try to make things that help them, you may find you're ahead on the trade. You'll be more constrained in what you can make, but you'll make it with more energy.

On the other hand, if you make something amazing, you'll often be helping people or the world even if you didn't mean to. Newton was driven by curiosity and ambition, not by any practical effect his work might have, and yet the practical effect of his work has been enormous. And this seems the rule rather than the exception. So if you think you can make something amazing, you should probably just go ahead and do it.

"""


sentences = [s.strip() for s in context.split(".") if s.strip()]

logger.info(f"Total Sentences: {len(sentences)}")

INFO: Total Sentences: 74


STEP 2: Loading Context...



# Step 3 — Text Preprocessing

Each sentence undergoes normalization:

1. Lowercasing  
2. Punctuation removal  
3. Tokenization  
4. Stopword removal  

If raw sentence is:

$$
s = \{ w_1, w_2, \dots, w_k \}
$$

After preprocessing:

$$
s' = \{ w_i \mid w_i \notin \text{Stopwords} \}
$$

This reduces noise and improves vector representation quality.

In [95]:
stopwords = set([
    "is","the","to","and","in","of","a","from","when","does","not","must"
])

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stopwords]
    return tokens

# Step 4 — Vocabulary Construction

Vocabulary is defined as:

$$
V = \bigcup_{i=1}^{N} \text{preprocess}(s_i)
$$

Where:

- $V$ = set of unique words  
- $|V|$ = vocabulary size  

Each word is mapped to an index:

$$
word \rightarrow index
$$

This enables vector representation in:

$$
\mathbb{R}^{|V|}
$$

In [96]:
vocab = list(set(word for s in sentences for word in preprocess(s)))
word_to_index = {w:i for i,w in enumerate(vocab)}

logger.info(f"Vocabulary Size: {len(vocab)}")

INFO: Vocabulary Size: 444


# Step 5 — Manual TF-IDF Model

TF-IDF has two components.

---

## Term Frequency (TF)

$$
TF(t,d) = \frac{\text{count of term } t \text{ in document } d}{\text{total terms in } d}
$$

---

## Inverse Document Frequency (IDF)

$$
IDF(t) = \log \left( \frac{N + 1}{df(t) + 1} \right) + 1
$$

Where:

- $N$ = total number of documents  
- $df(t)$ = number of documents containing term $t$  

---

## TF-IDF Weight

$$
TFIDF(t,d) = TF(t,d) \times IDF(t)
$$

---

## L2 Normalization

To normalize magnitude:

$$
\hat{v} = \frac{v}{||v||_2}
$$

Where:

$$
||v||_2 = \sqrt{\sum_{i=1}^{|V|} v_i^2}
$$

Normalization ensures fair cosine comparison.

In [97]:
class TFIDF:
    
    def fit(self, documents):
        self.N = len(documents)
        self.idf = {}
        
        for word in vocab:
            df = sum(1 for doc in documents if word in preprocess(doc))
            self.idf[word] = np.log((self.N + 1) / (df + 1)) + 1
    
    def transform(self, doc):
        tokens = preprocess(doc)
        tf = Counter(tokens)
        total = len(tokens)
        vector = np.zeros(len(vocab))
        
        for word in tf:
            if word in word_to_index and total > 0:
                vector[word_to_index[word]] = (tf[word] / total) * self.idf[word]
        
        norm = np.linalg.norm(vector)
        if norm != 0:
            vector = vector / norm
        
        return vector

tfidf = TFIDF()
tfidf.fit(sentences)

sentence_vectors = np.array([tfidf.transform(s) for s in sentences])

logger.info("Manual TF-IDF Model Trained.")

INFO: Manual TF-IDF Model Trained.


# Step 6 — Cosine Similarity

Similarity between two vectors $A$ and $B$:

$$
\cos(\theta) = \frac{A \cdot B}{||A|| \, ||B||}
$$

Where:

- $A \cdot B$ = dot product  
- $||A||$ = vector magnitude  

Range:

$$
0 \leq \cos(\theta) \leq 1
$$

Higher value indicates greater semantic similarity.

In [98]:
def cosine_similarity(vec1, vec2):
    if np.linalg.norm(vec1)==0 or np.linalg.norm(vec2)==0:
        return 0
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1)*np.linalg.norm(vec2))

# Step 7 — Retrieval Process

For a query $q$:

1. Transform into TF-IDF vector  
2. Compute similarity with each sentence  
3. Select highest similarity  

Mathematically:

$$
\hat{s} = \arg\max_{i} \cos(q, s_i)
$$

Where:

- $\hat{s}$ = predicted sentence  
- $s_i$ = candidate sentence  

In [99]:
def retrieve(question):
    q_vec = tfidf.transform(question)
    sims = [cosine_similarity(q_vec, s_vec) for s_vec in sentence_vectors]
    return np.argmax(sims), sims

# Step 8 — Evaluation Dataset

For each sentence $s_i$, we create a query:

$$
q_i = \text{first words of } s_i
$$

Ground truth label:

$$
y_i = i
$$

This creates a controlled self-retrieval evaluation setting.

In [100]:
questions = []
labels = []

for i, sentence in enumerate(sentences):
    short_query = " ".join(sentence.split()[:4])
    questions.append(short_query)
    labels.append(i)

logger.info("Evaluation Dataset Created.")

INFO: Evaluation Dataset Created.


# Step 9 — Accuracy

Accuracy is defined as:

$$
Accuracy = \frac{\sum I(y_i = \hat{y}_i)}{N}
$$

Where:

- $I$ = indicator function  
- $y_i$ = true label  
- $\hat{y}_i$ = predicted label  
- $N$ = total queries  

In [101]:
y_true = []
y_pred = []

for q, label in zip(questions, labels):
    pred, _ = retrieve(q)
    y_true.append(label)
    y_pred.append(pred)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

accuracy = np.mean(y_true == y_pred)
logger.info(f"Manual Accuracy: {accuracy:.4f}")

INFO: Manual Accuracy: 0.9189


# Step 11 — Ranking Metrics

Retrieval evaluates ranking quality.

---

## Mean Reciprocal Rank (MRR)

$$
MRR = \frac{1}{Q} \sum_{i=1}^{Q} \frac{1}{rank_i}
$$

---

## Precision@K

$$
Precision@K = \frac{\text{Relevant documents in Top K}}{K}
$$

---

## Recall@K

$$
Recall@K = \frac{\text{Relevant documents in Top K}}{\text{Total Relevant}}
$$

---

## NDCG@K

Discounted cumulative gain:

$$
DCG = \frac{1}{\log_2(rank + 1)}
$$

Normalized form:

$$
NDCG = \frac{DCG}{IDCG}
$$

These metrics evaluate ranking performance beyond top-1 accuracy.

In [102]:
def get_ranked(similarities):
    return np.argsort(similarities)[::-1]

def compute_rank(similarities, true_index):
    ranked = get_ranked(similarities)
    return np.where(ranked == true_index)[0][0] + 1

def calculate_mrr():
    return np.mean([
        1/compute_rank(retrieve(q)[1], label)
        for q,label in zip(questions,labels)
    ])

def precision_at_k(k):
    return np.mean([
        1/k if label in get_ranked(retrieve(q)[1])[:k] else 0
        for q,label in zip(questions,labels)
    ])

def recall_at_k(k):
    return np.mean([
        1 if label in get_ranked(retrieve(q)[1])[:k] else 0
        for q,label in zip(questions,labels)
    ])

def ndcg_at_k(k):
    scores=[]
    for q,label in zip(questions,labels):
        ranked=get_ranked(retrieve(q)[1])
        if label in ranked[:k]:
            rank=np.where(ranked==label)[0][0]+1
            scores.append(1/np.log2(rank+1))
        else:
            scores.append(0)
    return np.mean(scores)

logger.info(f"MRR: {calculate_mrr():.4f}")
logger.info(f"Precision@3: {precision_at_k(3):.4f}")
logger.info(f"Recall@3: {recall_at_k(3):.4f}")
logger.info(f"NDCG@3: {ndcg_at_k(3):.4f}")

INFO: MRR: 0.9572
INFO: Precision@3: 0.3333
INFO: Recall@3: 1.0000
INFO: NDCG@3: 0.9683


# Step 12 — Sklearn TF-IDF Comparison

Sklearn implements optimized TF-IDF.

Pipeline:

$$
Text \rightarrow Tokenization \rightarrow TF \rightarrow IDF \rightarrow Normalization
$$

Differences from manual version may arise due to:

- Token patterns  
- Stopword handling  
- Built-in normalization  

In [103]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=None,
    token_pattern=r'\b\w+\b',
    norm=None
)

sklearn_vectors = vectorizer.fit_transform(sentences)

def sklearn_retrieve(question):
    q_vec = vectorizer.transform([question])
    sims = sklearn_cosine(q_vec, sklearn_vectors)[0]
    return np.argmax(sims), sims

accuracy_sklearn = np.mean([
    sklearn_retrieve(q)[0]==label
    for q,label in zip(questions,labels)
])

logger.info(f"Sklearn Accuracy: {accuracy_sklearn:.4f}")

INFO: Sklearn Accuracy: 0.9459


# Step 13 — Benchmark Summary

We compare:

- Manual TF-IDF  
- Sklearn TF-IDF  

Metrics include:

- Accuracy  
- MRR  
- Precision@3  
- Recall@3  
- NDCG@3  

This provides structured performance comparison.

In [104]:
benchmark_df = pd.DataFrame([
    {
        "Model":"Manual TF-IDF",
        "Accuracy":accuracy,
        "MRR":calculate_mrr(),
        "Precision@3":precision_at_k(3),
        "Recall@3":recall_at_k(3),
        "NDCG@3":ndcg_at_k(3)
    },
    {
        "Model":"Sklearn TF-IDF",
        "Accuracy":accuracy_sklearn
    }
])

display(benchmark_df)

,Model,Accuracy,MRR,Precision@3,Recall@3,NDCG@3
0,Manual TF-IDF,0.918919,0.957207,0.333333,1.0,0.968306
1,Sklearn TF-IDF,0.945946,NaN,NaN,NaN,NaN


# Final Step — Extractive Summarization

Sentence importance is computed as:

$$
Importance(s_i) = \sum_{j \ne i} \cos(s_i, s_j)
$$

Sentences with highest centrality are selected.

This resembles a simplified TextRank method.

Top-N ranked sentences form the extractive summary.

In [105]:
logger.info("\nGenerating Extractive Summaries...\n")

# Number of sentences to include in summary
TOP_N = 3

# Manual TF-IDF Summary

def generate_manual_summary(top_n=3):
    importance_scores = []
    
    for i in range(len(sentence_vectors)):
        sims = [
            cosine_similarity(sentence_vectors[i], sentence_vectors[j])
            for j in range(len(sentence_vectors))
            if i != j
        ]
        importance_scores.append(sum(sims))
    
    ranked_indices = np.argsort(importance_scores)[::-1]
    top_indices = sorted(ranked_indices[:top_n])  # keep original order
    
    summary = " ".join([sentences[i] + "." for i in top_indices])
    
    return summary



# Sklearn TF-IDF Summary

def generate_sklearn_summary(top_n=3):
    
    importance_scores = []
    
    dense_vectors = sklearn_vectors.toarray()
    
    for i in range(len(dense_vectors)):
        sims = sklearn_cosine(
            [dense_vectors[i]],
            dense_vectors
        )[0]
        
        sims[i] = 0  # remove self similarity
        importance_scores.append(sum(sims))
    
    ranked_indices = np.argsort(importance_scores)[::-1]
    top_indices = sorted(ranked_indices[:top_n])
    
    summary = " ".join([sentences[i] + "." for i in top_indices])
    
    return summary


# Generate summaries
manual_summary = generate_manual_summary(TOP_N)
sklearn_summary = generate_sklearn_summary(TOP_N)


print("\n================ SUMMARY OUTPUT ================\n")

print("Manual TF-IDF Summary:\n")
print(manual_summary)

print("\n-----------------------------------------------\n")

print("Sklearn TF-IDF Summary:\n")
print(sklearn_summary)

INFO: 
Generating Extractive Summaries...




================ SUMMARY OUTPUT ================

Manual TF-IDF Summary:

I can't prove that one should do this, any more than I can prove that one should help people or take care of the world. But there's nothing in it about taking care of the world or making new things, and that's a bit worrying, because it seems like this question should be a timeless one. But even if you're one of those, you should at least make sure that the new things you make don't net harm people or the world.

-----------------------------------------------

Sklearn TF-IDF Summary:

Taking care of people and the world are shoulds in the sense that they're one's duty, but making good new things is a should in the sense that this is how to live to one's full potential. But there's nothing in it about taking care of the world or making new things, and that's a bit worrying, because it seems like this question should be a timeless one. Indeed this pattern is so common that you can use it as a recipe: if you're ex